**Scenario16
A retail company processes daily sales transactions from multiple store locations. The data arrives in different formats and needs to be cleaned, validated, and aggregated for business reporting. Using a Delta Live Tables (DLT) pipeline, the raw data is ingested from cloud storage, transformed with quality checks, and stored in Delta tables for analytics dashboards, ensuring accuracy and reliability in near real-time.**


In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Bronze Layer
@dlt.table(
name="customers_raw"
)
def customers_raw():
return spark.read.table("pyspark_cata.source.customers")


# Silver Layer
@dlt.table(
name="customers_enr"
)
def customers_enr():

df = spark.read.table("customers_raw")

df = df.withColumn(
"dedup",
row_number().over(
Window.partitionBy("id")
.orderBy(desc("modifiedDate"))
)
)

return (
df.where(col("dedup") == 1)
.drop("dedup")
)


# Gold Layer - SCD Type 2
dlt.create_streaming_table(
name="customers_dim"
)

dlt.create_auto_cdc_flow(
target="customers_dim",
source="customers_enr",
keys=["id"],
sequence_by="modifiedDate",
stored_as_scd_type=2
)


**If sequence_by throws an error, use:**
 
dlt.create_auto_cdc_flow(
target="customers_dim",
source="customers_enr",
keys=["id"],
sequence_by=col("modifiedDate"),
stored_as_scd_type=2
)
Notes
•	customers_raw reads source data.
•	customers_enr removes duplicates by keeping the latest record per id based on modifiedDate.
•	customers_dim is the target dimension table.
•	create_auto_cdc_flow() automatically manages: 
o	Insertions
o	Updates
o	Deletes (if configured in source)
o	History tracking using SCD Type 2
If you're using the latest Databricks Lakeflow Pipelines, an even better practice is:
from pyspark import pipelines as dp
and use dp.create_auto_cdc_flow(), since some environments are migrating from dlt to dp. However, the code above matches exactly what is shown in your screenshots.


